In [1]:
#Non Linear Plan: Pembuatan Kopi
class Action:
    def __init__(self, name, preconds, add_effects, del_effects):
        self.name = name
        self.preconds = set(preconds)
        self.add_effects = set(add_effects)
        self.del_effects = set(del_effects)
    def __repr__(self):
        return self.name

In [2]:
#Definisi Action Pembuatan Kopi
ACTIONS = [
    Action("Pickup(spoon)",
           ["ontable(spoon)", "handempty"],
           ["holding(spoon)"],
           ["ontable(spoon)", "handempty"]),

    Action("Add(coffee_powder,cup)",
           ["holding(spoon)", "ontable(coffee_powder)", "empty(cup)"],
           ["in(cup,coffee_powder)"],
           ["empty(cup)"]),

    Action("Add(sugar,cup)",
           ["holding(spoon)", "ontable(sugar)", "in(cup,coffee_powder)"],
           ["in(cup,sugar)"],
           []),

    Action("Putdown(spoon)",
           ["holding(spoon)"],
           ["ontable(spoon)", "handempty"],
           ["holding(spoon)"]),

    Action("Pickup(hot_water)",
           ["ontable(hot_water)", "handempty"],
           ["holding(hot_water)"],
           ["ontable(hot_water)", "handempty"]),

    Action("Pour(hot_water,cup)",
           ["holding(hot_water)", "in(cup,coffee_powder)", "in(cup,sugar)"],
           ["in(cup,hot_water)"],
           []),

    Action("Putdown(hot_water)",
           ["holding(hot_water)"],
           ["ontable(hot_water)", "handempty"],
           ["holding(hot_water)"]),

    # Stir is the operator the original domain was missing. Without it nothing
    # produces mixed(cup), so the goal was unreachable and no plan could ever
    # satisfy it - the old code injected the literal by hand instead.
    Action("Stir(cup)",
           ["holding(spoon)", "in(cup,coffee_powder)", "in(cup,sugar)",
            "in(cup,hot_water)"],
           ["mixed(cup)"],
           []),
]


In [3]:
#Kondisi Awal & Tujuan
INITIAL_STATE = {
    "ontable(spoon)", "ontable(coffee_powder)", "ontable(sugar)",
    "ontable(hot_water)", "empty(cup)", "handempty"
}
GOAL_STATE = {"mixed(cup)", "handempty"}

In [4]:
#Fungsi Bantu
def applicable(action, state):
    return all(p in state for p in action.preconds)
def apply(action, state):
    new_state = set(state)
    for d in action.del_effects:
        new_state.discard(d)
    new_state.update(action.add_effects)
    return new_state

In [5]:
#Algoritma Nonlinear Planning
from collections import deque


def search_plan(initial_state, goal_state, actions, max_depth=12):
    """Forward state-space search (breadth-first) for a shortest plan.

    Returns the list of actions, or None if the goal is unreachable. BFS is
    adequate here because the domain has 8 operators and a shallow solution;
    the point is that the plan is FOUND, not written out in advance.
    """
    start = frozenset(initial_state)
    goal = frozenset(goal_state)
    if goal <= start:
        return []

    seen = {start}
    queue = deque([(start, [])])
    while queue:
        state, plan = queue.popleft()
        if len(plan) >= max_depth:
            continue
        for a in actions:
            if not applicable(a, state):
                continue
            nxt = frozenset(apply(a, state))
            if nxt in seen:
                continue
            if goal <= nxt:
                return plan + [a]
            seen.add(nxt)
            queue.append((nxt, plan + [a]))
    return None


def causal_links(plan, initial_state):
    """Which step established each precondition of each later step.

    This is the partial order: a link j -> i means step j produced something
    step i needs, so j must precede i. Preconditions already true at the start
    produce no ordering constraint.
    """
    links = []
    for i, step in enumerate(plan):
        for p in sorted(step.preconds):
            producer = None
            for j in range(i - 1, -1, -1):
                if p in plan[j].add_effects:
                    producer = j
                    break
            if producer is not None:
                links.append((producer, i, p))
            elif p not in initial_state:
                links.append((None, i, p))      # unsupported - should not happen
    return links


def threats(plan, links):
    """A step between a link's producer and consumer that deletes the condition.

    Resolving these is what 'declobbering' means: the threatening step has to be
    promoted before the producer or demoted after the consumer.
    """
    found = []
    for j, i, p in links:
        if j is None:
            continue
        for k in range(j + 1, i):
            if p in plan[k].del_effects:
                found.append((k, j, i, p))
    return found


def nonlinear_plan(initial_state, goal_state, actions, verbose=True):
    """Find a plan, report its partial order, and verify the goal is achieved.

    Returns (plan, final_state, achieved) so the result can be used by other
    code - the previous version printed and returned None.
    """
    plan = search_plan(initial_state, goal_state, actions)

    if plan is None:
        reachable = set().union(*(a.add_effects for a in actions)) | set(initial_state)
        missing = set(goal_state) - reachable
        if verbose:
            print("=== NO PLAN FOUND ===")
            print(f"unreachable goal literals: {sorted(missing) or 'none'}")
            print("(a literal no operator produces can never be achieved)")
        return None, set(initial_state), False

    state = set(initial_state)
    if verbose:
        print("=== NONLINEAR PLAN: PEMBUATAN KOPI ===")
    for i, a in enumerate(plan, start=1):
        assert applicable(a, state), f"step {a} is not applicable - search is wrong"
        state = apply(a, state)
        if verbose:
            print(f"{i}. {a}")

    links = causal_links(plan, set(initial_state))
    if verbose:
        print("\n=== CAUSAL LINKS (derived, not hardcoded) ===")
        for j, i, p in links:
            print(f"- {plan[j]} must precede {plan[i]}   (provides {p})")

        th = threats(plan, links)
        print("\n=== THREATS / DECLOBBERING ===")
        if th:
            for k, j, i, p in th:
                print(f"- {plan[k]} deletes {p}, needed between {plan[j]} and {plan[i]}")
        else:
            print("- none: no step deletes a condition another step still needs")

    achieved = set(goal_state) <= state
    if verbose:
        print("\n=== GOAL CHECK ===")
        print(f"goal      : {sorted(goal_state)}")
        print(f"achieved  : {sorted(set(goal_state) & state)}")
        print(f"missing   : {sorted(set(goal_state) - state) or 'none'}")
        print("GOAL STATE TERCAPAI" if achieved else "GOAL NOT ACHIEVED")

    return plan, state, achieved


In [6]:
#Eksekusi
plan, final_state, achieved = nonlinear_plan(INITIAL_STATE, GOAL_STATE, ACTIONS)
print()
print(f"plan length: {len(plan) if plan else 0} steps   goal achieved: {achieved}")


=== NONLINEAR PLAN: PEMBUATAN KOPI ===
1. Pickup(spoon)
2. Add(coffee_powder,cup)
3. Add(sugar,cup)
4. Putdown(spoon)
5. Pickup(hot_water)
6. Pour(hot_water,cup)
7. Putdown(hot_water)
8. Pickup(spoon)
9. Stir(cup)
10. Putdown(spoon)

=== CAUSAL LINKS (derived, not hardcoded) ===
- Pickup(spoon) must precede Add(coffee_powder,cup)   (provides holding(spoon))
- Pickup(spoon) must precede Add(sugar,cup)   (provides holding(spoon))
- Add(coffee_powder,cup) must precede Add(sugar,cup)   (provides in(cup,coffee_powder))
- Pickup(spoon) must precede Putdown(spoon)   (provides holding(spoon))
- Putdown(spoon) must precede Pickup(hot_water)   (provides handempty)
- Pickup(hot_water) must precede Pour(hot_water,cup)   (provides holding(hot_water))
- Add(coffee_powder,cup) must precede Pour(hot_water,cup)   (provides in(cup,coffee_powder))
- Add(sugar,cup) must precede Pour(hot_water,cup)   (provides in(cup,sugar))
- Pickup(hot_water) must precede Putdown(hot_water)   (provides holding(hot_water)

In [7]:
#Demonstrasi: tanpa Stir, goal tidak tercapai
# Removing the Stir operator reproduces the original domain, in which no action
# produces mixed(cup). A correct planner must report failure rather than claim
# success - which is what the previous version did.
no_stir = [a for a in ACTIONS if a.name != "Stir(cup)"]
plan_x, state_x, achieved_x = nonlinear_plan(INITIAL_STATE, GOAL_STATE, no_stir)
print()
print(f"without Stir -> plan: {plan_x}   achieved: {achieved_x}")


=== NO PLAN FOUND ===
unreachable goal literals: ['mixed(cup)']
(a literal no operator produces can never be achieved)

without Stir -> plan: None   achieved: False


## Note on this notebook

The original version of this notebook did not plan. `ordered_steps` was a
hardcoded list of seven action names, the ordering constraints were a hardcoded
list of pairs, the goal literal `mixed(cup)` was inserted into the state by hand
(`state.add("mixed(cup)")`), and the final check printed the goal it had been
*given* rather than the state it reached - so it reported success unconditionally.

Three things changed:

1. **A real search.** `search_plan` does breadth-first forward state-space search
   and returns the shortest plan it finds, or `None` if the goal is unreachable.
2. **A `Stir(cup)` operator.** Nothing in the original domain produced
   `mixed(cup)`, so the goal was genuinely unreachable; the hand-inserted literal
   was hiding that. `Stir` requires a held spoon and all three ingredients.
3. **Derived ordering, and a real goal check.** Causal links and threats are
   computed from the plan that search returns, and `nonlinear_plan` now returns
   `(plan, final_state, achieved)` so the result can be used programmatically.

The last cell removes `Stir` again to show the planner correctly reporting
failure on the original domain.
